# H7 · Prediction: do country features help predict ROA in a country the model has never seen?

> **Hypothesis H7:** Country features improve predictions of company ROA **for countries left out of the training data**, beyond what sector and relative size already give.

| | |
|---|---|
| **Why we ask** | Everything so far asks whether there is an *association*. A harder, more practical question is whether it is useful: could an analyst predict a firm's profitability in a new country better by knowing that country's conditions? A clean "no" is itself a publishable result. |
| **Prediction** | **Positive** gain: lower prediction error than the sector-and-size baseline when whole countries are held out. |
| **Data** | Yahoo company-years (10,220). Baseline model: sector + relative size + fiscal year. Full model: the same plus the 35 previous-year country features. Ridge regression, **leave-one-country-out** validation (34 folds). |
| **Primary test (fixed before looking at the result)** | **Gain in mean absolute error (MAE)** = baseline MAE − full-model MAE, in percentage points of ROA, averaged over the 34 held-out countries. The 95% interval and one-sided p-value come from a bootstrap over countries. |
| **Verdict rule** | *Supported* if the effect has the predicted sign, the wild-bootstrap p < 0.05, and the sign holds in at least 80% of the robustness samples. *Partly supported* if it has the predicted sign and p < 0.10 (or p < 0.05 but robustness is below 80%). *Contradicted* if significantly the opposite way. Otherwise *not supported*. *For H7 the 'effect' is the MAE gain and the p-value is the bootstrap probability that the gain is ≤ 0; robustness is the share of alternative models (gradient boosting, fewer features, development index only, institutions only) that also beat the baseline.* |

**Honest framing.** Several hypotheses in this series (H1 to H3) were suggested by an exploratory scan of the same data, so their p-values are optimistic and a pass is a lead, not a confirmation. Where possible the result is **replicated on ESEF**, an independent source (which still shares some companies with Yahoo). All hypotheses are corrected for multiple testing together in `1.0_hypotheses.ipynb`. Everything is **association, not causation**.

In [1]:
import sys, pathlib, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore")
ROOT = pathlib.Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src" / "analysis"))
import prep, utils
from utils import CONTROLS, fit_cluster, wild_cluster, spearman, boot_ci, partial_rho, bh, scan

con = prep.connect()
q = lambda sql: con.sql(sql).df()
pd.set_option("display.width", 220, "display.max_columns", 40, "display.max_rows", 120, "display.float_format", "{:,.3f}".format)
px.defaults.template = "plotly_white"

country = prep.load_country(con)
dim, profile, profile_all, panel = country["dim"], country["profile"], country["profile_all"], country["panel"]
feat_cols = country["feat_cols"]
fy = prep.load_firms(con, country)               # Yahoo company-years (primary sample)
names = dim.name
MIN_FIRMS = prep.MIN_FIRMS
print(f"{len(dim)} countries | {len(feat_cols)} country features | {len(fy):,} Yahoo company-years of {fy.company_id.nunique():,} companies")

SAMPLES = {
 "all firms": lambda d: d,
 "excluding financials & real estate": lambda d: d[~d.sector.isin(["Financial Services", "Real Estate"])],
 "excluding small-sample countries": lambda d: d[~d.iso3.isin(dim.index[dim.small_firm_sample])],
 "excluding USA, China, Japan": lambda d: d[~d.iso3.isin(["USA", "CHN", "JPN"])],
 "fiscal years 2023 and later": lambda d: d[d.fiscal_year >= 2023],
 "EU countries only": lambda d: d[d.iso3.isin(dim.index[dim.is_eu])],
 "non-EU countries only": lambda d: d[~d.iso3.isin(dim.index[dim.is_eu])],
}
def robustness(focal, outcome, expected_sign, B=499, extra=""):
    """Re-estimate the primary test in each sample. Returns a table and the share of samples with the expected sign."""
    rows = []
    for name, flt in SAMPLES.items():
        r = wild_cluster(focal, flt(fy), outcome=outcome, extra=extra, B=B)
        rows.append({"sample": name, "coef": r["coef"], "ci_low": r["ci_low"], "ci_high": r["ci_high"], "p (wild)": r["p_wild"],
                     "countries": r["clusters"], "firm-years": r["n"]})
    t = pd.DataFrame(rows).set_index("sample")
    same = (np.sign(t.coef) == expected_sign) if expected_sign else pd.Series(True, index=t.index)
    return t.assign(**{"expected sign": same}), float(same.mean())

from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold, KFold

34 countries | 35 country features | 10,220 Yahoo company-years of 2,590 companies


## 1 · Set-up
**Leave-one-country-out:** for each of the 34 countries, the model is trained on the other 33 and predicts the held-out country's companies. This is the honest test of "a country the model has never seen". A random split of company-years would be misleading, because firms in the same country (and the same firm in different years) leak information to each other.

Models, all predicting ROA in percentage points:
- **Global median** (a constant) and **sector median**: the weakest baselines.
- **Baseline:** Ridge on sector + relative size + fiscal year.
- **Full:** the baseline plus the 35 previous-year country features (standardised; missing values filled with the training median).

In [2]:
feats_z = [f + "_z" for f in feat_cols]
d = fy.dropna(subset=["roa_pp", "size_rank"]).reset_index(drop=True)
cat, num_base = ["sector"], ["size_rank"]
y = d.roa_pp.values
groups = d.iso3.values
print(f"{len(d):,} company-years, {d.iso3.nunique()} countries; ROA median {np.median(y):.1f} pp, SD {y.std():.1f} pp")

d["fiscal_year"] = d.fiscal_year.astype(str)
cat_all = ["sector", "fiscal_year"]
def make(num_cols, model):
    pre = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), cat_all),
                             ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num_cols)])
    return Pipeline([("pre", pre), ("m", model)])

MODELS = {
 "baseline (sector + size + year)": lambda: make(num_base, Ridge(alpha=10)),
 "full (+ 35 country features)": lambda: make(num_base + feats_z, Ridge(alpha=10)),
 "+ development index only": lambda: make(num_base + ["dev_z"], Ridge(alpha=10)),
 "+ institutions index only": lambda: make(num_base + ["inst_z"], Ridge(alpha=10)),
 "+ 8 representative features": lambda: make(num_base + [f + "_z" for f in ["gdp_per_capita_ppp", "rule_of_law", "trade_gdp", "rd_gdp", "unemployment", "private_credit_gdp", "inflation", "pop_65plus"]], Ridge(alpha=10)),
 "full, gradient boosting": lambda: make(num_base + feats_z, HistGradientBoostingRegressor(max_depth=3, learning_rate=0.05, max_iter=200, random_state=0)),
}
def loco(model_fn):
    pred = np.zeros(len(d))
    for c in np.unique(groups):
        te = groups == c
        m = model_fn().fit(d.loc[~te], y[~te])
        pred[te] = m.predict(d.loc[te])
    return pred

10,220 company-years, 34 countries; ROA median 3.8 pp, SD 8.9 pp


## 2 · Out-of-country predictions

In [3]:
preds = {}
for name, fn in MODELS.items():
    preds[name] = loco(fn)
# constant baselines
cmed = np.zeros(len(d)); smed = np.zeros(len(d))
for c in np.unique(groups):
    te = groups == c
    cmed[te] = np.median(y[~te])
    sm = d.loc[~te].groupby("sector").roa_pp.median()
    smed[te] = d.loc[te, "sector"].map(sm).fillna(np.median(y[~te])).values
preds = {"global median": cmed, "sector median": smed, **preds}
err = pd.DataFrame({k: np.abs(y - v) for k, v in preds.items()}); err["iso3"] = groups
by_country = err.groupby("iso3").mean()                         # MAE per held-out country
res = pd.DataFrame({"MAE (pp)": {k: np.abs(y - v).mean() for k, v in preds.items()},
                    "MAE, mean over countries (pp)": by_country.mean(),
                    "R² (out of country)": {k: 1 - np.sum((y - v) ** 2) / np.sum((y - y.mean()) ** 2) for k, v in preds.items()}})
res

,MAE (pp),"MAE, mean over countries (pp)",R² (out of country)
global median,5.471,5.477,-0.003
sector median,5.281,5.329,0.008
baseline (sector + size + year),5.451,5.470,0.012
full (+ 35 country features),5.857,5.850,-0.036
+ development index only,5.434,5.458,0.028
+ institutions index only,5.455,5.472,0.026
+ 8 representative features,5.588,5.613,0.012
"full, gradient boosting",5.346,5.393,0.071


## 3 · Primary test: does the full model beat the baseline?
Per-country MAE difference (baseline − full): positive means the country features helped for that country. The interval and p-value come from resampling the 34 countries (5,000 bootstraps).

In [4]:
bn, fn_ = "baseline (sector + size + year)", "full (+ 35 country features)"
gain = (by_country[bn] - by_country[fn_])
rng = np.random.default_rng(0)
boots = np.array([gain.values[rng.integers(0, len(gain), len(gain))].mean() for _ in range(5000)])
g_mean, (g_lo, g_hi) = gain.mean(), np.percentile(boots, [2.5, 97.5])
p_gain = float((np.sum(boots <= 0) + 1) / (len(boots) + 1))
wins = int((gain > 0).sum())
print(f"MAE gain from adding the 35 country features: {g_mean:+.3f} pp ROA (95% CI {g_lo:+.3f} to {g_hi:+.3f}); one-sided bootstrap p (gain ≤ 0) = {p_gain:.3f}")
print(f"The full model is better in {wins} of {len(gain)} held-out countries. Baseline MAE {by_country[bn].mean():.2f} pp, full {by_country[fn_].mean():.2f} pp (typical ROA {np.median(y):.1f} pp).")
g = gain.sort_values().rename(lambda i: names[i])
fig = px.bar(g, orientation="h", title="MAE gain per held-out country (baseline − full, pp ROA; positive = country features helped)", labels={"value": "gain (pp)", "index": ""}, height=760)
fig.add_vline(x=0, line_color="grey"); fig.update_layout(showlegend=False); fig.show()

MAE gain from adding the 35 country features: -0.380 pp ROA (95% CI -0.609 to -0.187); one-sided bootstrap p (gain ≤ 0) = 1.000
The full model is better in 9 of 34 held-out countries. Baseline MAE 5.47 pp, full 5.85 pp (typical ROA 3.8 pp).


## 4 · Robustness: other models and feature sets
The same test for the alternative models. The robustness share is the share of alternatives with a positive mean gain over the baseline.

In [5]:
rows = []
for name in [k for k in MODELS if k != bn]:
    gn = by_country[bn] - by_country[name]
    bs = np.array([gn.values[rng.integers(0, len(gn), len(gn))].mean() for _ in range(2000)])
    rows.append({"model": name, "mean MAE gain (pp)": gn.mean(), "ci_low": np.percentile(bs, 2.5), "ci_high": np.percentile(bs, 97.5),
                 "countries improved": f"{int((gn > 0).sum())} of {len(gn)}", "p (gain ≤ 0)": (np.sum(bs <= 0) + 1) / (len(bs) + 1)})
alt = pd.DataFrame(rows).set_index("model")
rob_share = float((alt["mean MAE gain (pp)"] > 0).mean())
print(f"Positive mean gain in {rob_share:.0%} of {len(alt)} alternative models.")
alt

Positive mean gain in 40% of 5 alternative models.


,mean MAE gain (pp),ci_low,ci_high,countries improved,p (gain ≤ 0)
model,,,,,
full (+ 35 country features),-0.380,-0.600,-0.178,9 of 34,1.000
+ development index only,0.012,-0.042,0.068,16 of 34,0.338
+ institutions index only,-0.002,-0.056,0.049,17 of 34,0.533
+ 8 representative features,-0.144,-0.262,-0.035,18 of 34,0.996
"full, gradient boosting",0.077,-0.026,0.183,19 of 34,0.074


### A contrast: ordinary random split
The same models evaluated with a random 5-fold split of company-years, so companies from the same country (and the same firm in other years) appear in both training and test data. This is the usual but misleading way to validate; here it is shown only for comparison.

In [6]:
kf = KFold(5, shuffle=True, random_state=0)
pr_b = np.zeros(len(d)); pr_f = np.zeros(len(d))
for tr, te in kf.split(d):
    pr_b[te] = MODELS[bn]().fit(d.iloc[tr], y[tr]).predict(d.iloc[te])
    pr_f[te] = MODELS[fn_]().fit(d.iloc[tr], y[tr]).predict(d.iloc[te])
leak = pd.DataFrame({"validation": ["random split (leaky)", "leave-one-country-out"], "baseline MAE (pp)": [np.abs(y - pr_b).mean(), res.loc[bn, "MAE (pp)"]],
                     "full MAE (pp)": [np.abs(y - pr_f).mean(), res.loc[fn_, "MAE (pp)"]]})
leak["gain (pp)"] = leak["baseline MAE (pp)"] - leak["full MAE (pp)"]
leak.set_index("validation")

,baseline MAE (pp),full MAE (pp),gain (pp)
validation,,,
random split (leaky),5.410,5.403,0.006
leave-one-country-out,5.451,5.857,-0.406


## 5 · Replication on ESEF
The same leave-one-country-out comparison on the independent EU source (no sector, so the baseline is size + year only).

In [7]:
fe = prep.load_firms(con, country, source="ESEF").dropna(subset=["roa_pp", "size_rank"]).reset_index(drop=True)
fe["fiscal_year"] = fe.fiscal_year.astype(str)
ye, ge_ = fe.roa_pp.values, fe.iso3.values
def make_e(num_cols):
    pre = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), ["fiscal_year"]),
                             ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num_cols)])
    return Pipeline([("pre", pre), ("m", Ridge(alpha=10))])
res_e = {}
for name, cols in {"base": ["size_rank"], "full": ["size_rank"] + feats_z}.items():
    pr = np.zeros(len(fe))
    for c in np.unique(ge_):
        te = ge_ == c
        pr[te] = make_e(cols).fit(fe.loc[~te], ye[~te]).predict(fe.loc[te])
    res_e[name] = pd.Series(np.abs(ye - pr)).groupby(ge_).mean()
gain_e = res_e["base"] - res_e["full"]
bs_e = np.array([gain_e.values[rng.integers(0, len(gain_e), len(gain_e))].mean() for _ in range(2000)])
p_e = float((np.sum(bs_e <= 0) + 1) / (len(bs_e) + 1))
print(f"ESEF ({len(fe):,} company-years, {fe.iso3.nunique()} countries): MAE gain from country features {gain_e.mean():+.3f} pp (95% CI {np.percentile(bs_e, 2.5):+.3f} to {np.percentile(bs_e, 97.5):+.3f}), p (gain ≤ 0) = {p_e:.3f}; better in {int((gain_e > 0).sum())} of {len(gain_e)} countries.")

ESEF (5,312 company-years, 19 countries): MAE gain from country features -1.215 pp (95% CI -2.125 to -0.489), p (gain ≤ 0) = 1.000; better in 2 of 19 countries.


## 6 · Verdict

In [8]:
v = utils.verdict(g_mean, p_gain, +1, rob_share, int(len(gain)))
lines = [
 f"Primary: adding the 35 country features changes out-of-country MAE by {g_mean:+.3f} pp ROA (95% CI {g_lo:+.3f} to {g_hi:+.3f}); better in {wins} of {len(gain)} countries; p (gain ≤ 0) = {p_gain:.3f}.",
 f"Context: baseline MAE {by_country[bn].mean():.2f} pp vs a typical ROA of {np.median(y):.1f} pp; out-of-country R² {res.loc[bn, 'R² (out of country)']:.3f} (baseline) vs {res.loc[fn_, 'R² (out of country)']:.3f} (full).",
 f"Robustness: {rob_share:.0%} of {len(alt)} alternative models have a positive gain. For comparison, an ordinary random split (not a valid test for new countries) gives a gain of {leak.iloc[0]['gain (pp)']:+.3f} pp.",
  f"Note: the sector median (MAE {res.loc['sector median', 'MAE (pp)']:.2f} pp) beats the Ridge baseline ({res.loc[bn, 'MAE (pp)']:.2f} pp) on MAE, because Ridge minimises squared error and ROA has fat tails; the comparison that matters here is full vs baseline, with the same loss.",
 f"ESEF replication: gain {gain_e.mean():+.3f} pp (p = {p_e:.3f}).",
 f"VERDICT H7: {v.upper()}",
]
print("\n".join(lines))
utils.save_result("H7", id="H7", title="Out-of-country prediction", statement="Country features improve ROA predictions for countries the model has not seen.",
                  expected_sign=1, effect=float(g_mean), ci_low=float(g_lo), ci_high=float(g_hi), effect_unit="MAE gain in pp ROA (baseline − full), leave-one-country-out",
                  p_primary=p_gain, p_method="bootstrap over countries (gain ≤ 0)", n=int(len(d)), clusters=int(len(gain)), robust_share=rob_share,
                  replication_coef=float(gain_e.mean()), replication_p=p_e, replication_same_sign=bool(gain_e.mean() > 0), verdict=v,
                  notebook="1.7_out_of_country_prediction.ipynb", notes=lines)

Primary: adding the 35 country features changes out-of-country MAE by -0.380 pp ROA (95% CI -0.609 to -0.187); better in 9 of 34 countries; p (gain ≤ 0) = 1.000.
Context: baseline MAE 5.47 pp vs a typical ROA of 3.8 pp; out-of-country R² 0.012 (baseline) vs -0.036 (full).
Robustness: 40% of 5 alternative models have a positive gain. For comparison, an ordinary random split (not a valid test for new countries) gives a gain of +0.006 pp.
Note: the sector median (MAE 5.28 pp) beats the Ridge baseline (5.45 pp) on MAE, because Ridge minimises squared error and ROA has fat tails; the comparison that matters here is full vs baseline, with the same loss.
ESEF replication: gain -1.215 pp (p = 1.000).
VERDICT H7: NOT SUPPORTED


### Limits
- Firm-level ROA is noisy, so even a real country effect may add little to prediction; the baseline MAE shows how large the unexplained noise is.
- Ridge and a small gradient-boosting model were used with fixed settings (no tuning), to keep the test simple and avoid over-fitting the test countries.
- Previous-year country features for fiscal years 2025 and later are partly missing and filled with the training median.
- The ESEF replication has no sector field.
- This predicts, it does not explain: association, not causation.

In [9]:
con.close()